In [3]:
with open("cases_all_updated.json", encoding="utf-8") as f:
        backup_cases_all = json.load(f)
backup_cases_all.keys()

dict_keys(['[1998] HCA 28', '[2003] HCA 2', '[2003] HCA 22', '[2001] HCA 30', '[2009] HCA 27', '[2005] HCA 25', '[2010] HCA 16', '[1998] HCA 11', '[2000] HCA 63', '[2006] HCA 63', '[1999] HCA 14', '[2001] HCA 17', '[2000] HCA 54', '[1998] HCA 57', '[2010] HCA 45', '[2004] HCA 52', '[2009] HCA 41', '[2001] HCA 64', '[1999] HCA 21', '[2000] HCA 57', '[2011] HCA 39', '[2000] HCA 47', '[2010] HCA 28', '[2007] HCA 22', '[2010] HCA 1', '[2005] HCA 24', '[2003] HCA 6', '[2000] HCA 48', '[2010] HCA 4', '[1998] HCA 67', '[1999] HCA 54', '[2000] HCA 40', '[2008] HCA 31', '[1999] HCA 29', '[2011] HCA 49', '[2011] HCA 4', '[2002] HCA 11', '[2001] HCA 63', '[2004] HCA 35', '[2002] HCA 53', '[2006] HCA 27', '[2000] HCA 41', '[2002] HCA 36', '[1999] HCA 27', '[1998] HCA 27', '[1999] HCA 26', '[1999] HCA 66', '[2001] HCA 22', '[1999] HCA 67', '[2002] HCA 35', '[2011] HCA 13', '[2010] HCA 41', '[2011] HCA 48', '[2006] HCA 46', '[1998] HCA 55', '[1999] HCA 9', '[2011] HCA 21', '[1998] HCA 30', '[2007] H

In [6]:
import json
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
import networkx as nx
import numpy as np
from collections import defaultdict
from itertools import combinations

# --- Load & compute ---
data = backup_cases_all


co_authored = defaultdict(int)
judge_totals = defaultdict(int)

for case in data.values():
    for j in case['judgmentAuthors']:
        names = [n.strip() for n in j['name'].split(',')]
        for name in names:
            judge_totals[name] += 1
        for a, b in combinations(sorted(names), 2):
            co_authored[(a, b)] += 1

JUDGES = list(judge_totals.keys())

def build_graph(weight_fn):
    G = nx.Graph()
    G.add_nodes_from(JUDGES)
    for (a, b), count in co_authored.items():
        if a in JUDGES and b in JUDGES:
            w = weight_fn(a, b, count)
            if w > 0:
                G.add_edge(a, b, weight=w, raw=count)
    return G

def draw_network(G, title, filename):
    fig, ax = plt.subplots(figsize=(12, 9))
    fig.patch.set_facecolor('white')
    ax.set_facecolor('white')
    ax.axis('off')

    pos = nx.spring_layout(G, weight='weight', seed=42, k=2.8)

    weights = [G[u][v]['weight'] for u, v in G.edges()]
    w_min, w_max = min(weights), max(weights)

    # Draw edges — grey, varying width and alpha
    for u, v in G.edges():
        w = G[u][v]['weight']
        t = (w - w_min) / (w_max - w_min) if w_max > w_min else 1.0
        alpha = 0.15 + t * 0.75
        lw    = 0.4 + t * 5.5
        x0, y0 = pos[u]
        x1, y1 = pos[v]
        ax.plot([x0, x1], [y0, y1],
                color='#666666', alpha=alpha, linewidth=lw,
                zorder=1, solid_capstyle='round')

    # Node size proportional to total judgments
    max_total = max(judge_totals[j] for j in JUDGES)
    for judge in JUDGES:
        x, y = pos[judge]
        size = 300 + 1400 * (judge_totals[judge] / max_total)
        ax.scatter(x, y, s=size, color='#333333', zorder=3, edgecolors='#888888', linewidths=1.5)
        # Label offset upward from node centre
        offset = 0.055 + 0.04 * (judge_totals[judge] / max_total)
        label = f"{judge.capitalize()} ({judge_totals[judge]})"
        ax.text(x, y + offset, label,
                ha='center', va='bottom', fontsize=9,
                color='#111111', fontweight='bold', zorder=5)

    ax.set_title(title, fontsize=13, fontweight='bold', pad=14, color='#111111')

    # Legend
    for label, t in [('Weak', 0.05), ('Medium', 0.5), ('Strong', 1.0)]:
        lw    = 0.4 + t * 5.5
        alpha = 0.15 + t * 0.75
        ax.plot([], [], color='#666666', alpha=alpha, linewidth=lw, label=label)
    leg = ax.legend(loc='lower left', frameon=True, fontsize=9,
                    facecolor='white', edgecolor='#aaaaaa',
                    title='Connection strength', title_fontsize=9)

    plt.tight_layout()
    plt.savefig(filename, dpi=150, bbox_inches='tight', facecolor='white')
    plt.close()
    print(f"Saved: {filename}")

# Graph 1 — raw
G1 = build_graph(lambda a, b, count: count)
draw_network(G1,
    title="Judge Co-authorship Network (raw count)",
    filename='images/judge_network_raw.png')

# Graph 2 — normalised: 2*count / (total_A + total_B)
def norm_weight(a, b, count):
    return (2 * count) / (judge_totals[a] + judge_totals[b])

G2 = build_graph(norm_weight)
draw_network(G2,
    title="Judge Co-authorship Network (normalised by individual output)",
    filename='images/judge_network_normalised.png')

Saved: images/judge_network_raw.png
Saved: images/judge_network_normalised.png


In [ ]:

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from collections import defaultdict

data = backup_cases_all

judge_totals = defaultdict(int)
judge_collab = defaultdict(int)

for case in data.values():
    for j in case['judgmentAuthors']:
        names = [n.strip() for n in j['name'].split(',')]
        for name in names:
            judge_totals[name] += 1
            if len(names) > 1:
                judge_collab[name] += 1

judges   = sorted(judge_totals.keys())
totals   = [judge_totals[j] for j in judges]
collabs  = [judge_collab[j] for j in judges]
ratios   = [judge_collab[j] / judge_totals[j] for j in judges]

fig, ax = plt.subplots(figsize=(10, 7))
fig.patch.set_facecolor('white')
ax.set_facecolor('#f8f8f8')

ax.scatter(totals, ratios, s=80, color='#333333', zorder=3)

# Label each point
for judge, x, y in zip(judges, totals, ratios):
    ax.annotate(
        f"{judge.capitalize()}\n({judge_collab[judge]}/{judge_totals[judge]})",
        xy=(x, y),
        xytext=(6, 4),
        textcoords='offset points',
        fontsize=8.5,
        color='#222222',
    )

ax.set_xlabel('Total judgments written', fontsize=12)
ax.set_ylabel('Collaboration rate\n(joint judgments / total judgments)', fontsize=12)
ax.set_title('Judge Collaboration Rate vs Total Output', fontsize=14, fontweight='bold', pad=14)
ax.set_ylim(-0.05, 1.05)
ax.yaxis.set_major_formatter(matplotlib.ticker.PercentFormatter(xmax=1))
ax.grid(True, linestyle='--', alpha=0.5, zorder=0)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('images/collab_scatter.png', dpi=150, bbox_inches='tight', facecolor='white')
print("Saved: collab_scatter.png")

Saved: collab_scatter.png


In [8]:

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker
import numpy as np
from collections import defaultdict, Counter


data = backup_cases_all

together_raw      = defaultdict(int)
together_majority = defaultdict(int)
together_alone    = defaultdict(int)
tag_totals        = defaultdict(int)
gh_judgment_sizes = []

for case in data.values():
    tags       = case['law_areas']
    bench_size = len(case.get('bench', []))
    for tag in tags:
        tag_totals[tag] += 1

    gh_judgment = None
    for j in case['judgmentAuthors']:
        names = [n.strip() for n in j['name'].split(',')]
        if 'GUMMOW' in names and 'HAYNE' in names:
            gh_judgment = names
            break

    if gh_judgment:
        n_authors  = len(gh_judgment)
        is_majority = n_authors > bench_size / 2 if bench_size > 0 else False
        is_alone    = n_authors == 2
        for tag in tags:
            together_raw[tag] += 1
            if is_majority:
                together_majority[tag] += 1
            if is_alone:
                together_alone[tag] += 1
        gh_judgment_sizes.append(n_authors)

# Sort by raw count descending
tags_sorted  = sorted(tag_totals.keys(), key=lambda t: -together_raw.get(t, 0))
raw          = [together_raw.get(t, 0)      for t in tags_sorted]
normed       = [together_raw.get(t, 0) / tag_totals[t] for t in tags_sorted]
maj_of_tag   = [together_majority.get(t, 0) / tag_totals[t] for t in tags_sorted]
alone_of_tag = [together_alone.get(t, 0)    / tag_totals[t] for t in tags_sorted]
labels       = [t.replace(' & ', '\n& ').replace(', Planning', ',\nPlanning') for t in tags_sorted]
x            = np.arange(len(tags_sorted))

BAR_COLOR  = '#333333'
MAJ_COLOR  = '#cc3333'
ALONE_COLOR = '#3366cc'

# ══════════════════════════════════════════════════════════
# GRAPHS 1, 2 & 3 — stacked subplots
# ══════════════════════════════════════════════════════════
fig, axes = plt.subplots(3, 1, figsize=(16, 15), sharex=True,
                         gridspec_kw={'hspace': 0.12})
fig.patch.set_facecolor('white')

# --- Graph 1: raw counts ---
ax1 = axes[0]
ax1.set_facecolor('#f8f8f8')
ax1.bar(x, raw, color=BAR_COLOR, edgecolor='white', linewidth=0.6, zorder=3)
for i, v in enumerate(raw):
    if v > 0:
        ax1.text(i, v + 0.4, str(v), ha='center', va='bottom', fontsize=8, color='#222')
# n= labels inside top of each bar (or just below bar top)
for i, tag in enumerate(tags_sorted):
    ax1.text(i, 0.8, f'n={tag_totals[tag]}', ha='center', va='bottom',
             fontsize=6.5, color='#aaa', zorder=4)
ax1.set_ylabel('Cases written together', fontsize=11)
ax1.set_title('Gummow & Hayne — Cases co-authored by tag (raw)', fontsize=13, fontweight='bold', pad=10)
ax1.yaxis.grid(True, linestyle='--', alpha=0.5, zorder=0)
ax1.set_axisbelow(True)
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)

# --- Graph 2: normalised + majority overlay ---
ax2 = axes[1]
ax2.set_facecolor('#f8f8f8')
ax2.bar(x, normed,     color=BAR_COLOR, edgecolor='white', linewidth=0.6, zorder=3, label='Co-authorship rate')
ax2.bar(x, maj_of_tag, color=MAJ_COLOR, edgecolor='white', linewidth=0.6, zorder=4, alpha=0.75,
        label='Of which: majority judgment')
for i, v in enumerate(normed):
    if v > 0:
        ax2.text(i, v + 0.005, f'{v:.0%}', ha='center', va='bottom', fontsize=7.5, color='#222')
ax2.set_ylabel('Share of tag cases', fontsize=11)
ax2.set_title('Gummow & Hayne — Co-authorship rate by tag (normalised), with majority overlay',
              fontsize=13, fontweight='bold', pad=10)
ax2.yaxis.set_major_formatter(matplotlib.ticker.PercentFormatter(xmax=1))
ax2.yaxis.grid(True, linestyle='--', alpha=0.5, zorder=0)
ax2.set_axisbelow(True)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)
ax2.legend(fontsize=9, frameon=True, facecolor='white', edgecolor='#aaa')

# --- Graph 3: normalised + alone overlay ---
ax3 = axes[2]
ax3.set_facecolor('#f8f8f8')
ax3.bar(x, normed,       color=BAR_COLOR,  edgecolor='white', linewidth=0.6, zorder=3, label='Co-authorship rate')
ax3.bar(x, alone_of_tag, color=ALONE_COLOR, edgecolor='white', linewidth=0.6, zorder=4, alpha=0.75,
        label='Of which: Gummow & Hayne alone')
for i, v in enumerate(normed):
    if v > 0:
        ax3.text(i, v + 0.005, f'{v:.0%}', ha='center', va='bottom', fontsize=7.5, color='#222')
ax3.set_ylabel('Share of tag cases', fontsize=11)
ax3.set_title('Gummow & Hayne — Co-authorship rate by tag (normalised), with alone overlay',
              fontsize=13, fontweight='bold', pad=10)
ax3.yaxis.set_major_formatter(matplotlib.ticker.PercentFormatter(xmax=1))
ax3.yaxis.grid(True, linestyle='--', alpha=0.5, zorder=0)
ax3.set_axisbelow(True)
ax3.spines['top'].set_visible(False)
ax3.spines['right'].set_visible(False)
ax3.legend(fontsize=9, frameon=True, facecolor='white', edgecolor='#aaa')

ax3.set_xticks(x)
ax3.set_xticklabels(labels, rotation=40, ha='right', fontsize=8.5)

plt.tight_layout()
plt.savefig('images/gummow_hayne_by_tag.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.close()
print("Saved: gummow_hayne_by_tag.png")

# ══════════════════════════════════════════════════════════
# JUDGMENT SIZE graph (unchanged)
# ══════════════════════════════════════════════════════════
size_counts = Counter(gh_judgment_sizes)
sizes  = sorted(size_counts.keys())
counts = [size_counts[s] for s in sizes]

fig, ax = plt.subplots(figsize=(9, 6))
fig.patch.set_facecolor('white')
ax.set_facecolor('#f8f8f8')
bars = ax.bar(sizes, counts, color=BAR_COLOR, edgecolor='white', linewidth=0.8, zorder=3, width=0.6)
for bar, val in zip(bars, counts):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
            str(val), ha='center', va='bottom', fontsize=10, color='#222')
ax.set_xlabel('Total judges in the Gummow–Hayne judgment', fontsize=11)
ax.set_ylabel('Number of cases', fontsize=11)
ax.set_title('Gummow & Hayne — How many judges did they write with?',
             fontsize=13, fontweight='bold', pad=12)
ax.set_xticks(sizes)
ax.set_xticklabels([
    f'{s}\n(G+H {"alone" if s == 2 else f"+ {s-2} other{"s" if s-2 > 1 else ""}"})'
    for s in sizes
], fontsize=9)
ax.yaxis.grid(True, linestyle='--', alpha=0.5, zorder=0)
ax.set_axisbelow(True)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig('images/gummow_hayne_judgment_size.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.close()
print("Saved: gummow_hayne_judgment_size.png")

/var/folders/4k/yfp_3j857vq5fj717924mjfm0000gn/T/ipykernel_69860/4248368824.py:121: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()


Saved: gummow_hayne_by_tag.png
Saved: gummow_hayne_judgment_size.png


In [9]:
for case in backup_cases_all.values():
    judgments = case['judgmentAuthors']
    sorted_j = sorted(judgments, key=lambda j: j.get('totalCitations', 0), reverse=True)
    top_citations = sorted_j[0].get('totalCitations', 0) if sorted_j else 0
 
    for rank, j in enumerate(sorted_j, start=1):
        cites = j.get('totalCitations', 0)
        j['citationRank'] = rank
        j['relativeCitation'] = round(cites / top_citations, 4) if top_citations > 0 else 0
 
    case['judgmentAuthors'] = sorted_j

In [12]:
with open("cases_all_updated.json", "w", encoding="utf-8") as f:
    json.dump(backup_cases_all, f, indent=2, ensure_ascii=False)
    
with open("cases_all_updated.json", encoding="utf-8") as f:
        backup_cases_all = json.load(f)
backup_cases_all.keys()

dict_keys(['[1998] HCA 28', '[2003] HCA 2', '[2003] HCA 22', '[2001] HCA 30', '[2009] HCA 27', '[2005] HCA 25', '[2010] HCA 16', '[1998] HCA 11', '[2000] HCA 63', '[2006] HCA 63', '[1999] HCA 14', '[2001] HCA 17', '[2000] HCA 54', '[1998] HCA 57', '[2010] HCA 45', '[2004] HCA 52', '[2009] HCA 41', '[2001] HCA 64', '[1999] HCA 21', '[2000] HCA 57', '[2011] HCA 39', '[2000] HCA 47', '[2010] HCA 28', '[2007] HCA 22', '[2010] HCA 1', '[2005] HCA 24', '[2003] HCA 6', '[2000] HCA 48', '[2010] HCA 4', '[1998] HCA 67', '[1999] HCA 54', '[2000] HCA 40', '[2008] HCA 31', '[1999] HCA 29', '[2011] HCA 49', '[2011] HCA 4', '[2002] HCA 11', '[2001] HCA 63', '[2004] HCA 35', '[2002] HCA 53', '[2006] HCA 27', '[2000] HCA 41', '[2002] HCA 36', '[1999] HCA 27', '[1998] HCA 27', '[1999] HCA 26', '[1999] HCA 66', '[2001] HCA 22', '[1999] HCA 67', '[2002] HCA 35', '[2011] HCA 13', '[2010] HCA 41', '[2011] HCA 48', '[2006] HCA 46', '[1998] HCA 55', '[1999] HCA 9', '[2011] HCA 21', '[1998] HCA 30', '[2007] H

In [13]:
print(backup_cases_all["[2001] HCA 50"])

{'citation': '[2001] HCA 50', 'catchwords': 'Smith v The Queen Criminal law – Evidence – Relevance – Identification – Evidence of recognition by police officers of the accused in photographs from bank security cameras – Police officers in no better position than jury to compare appearance of accused with photographs – Evidence Act 1995 (NSW), s 55 – Whether evidence could rationally affect the assessment by jury of probability of the existence of a fact in issue – Whether admissible as opinion evidence. Evidence – Relevance – Opinion or fact evidence – Evidence Act 1995 (NSW), s 55 – Evidence of recognition by police officers of accused in photographs from bank security cameras – Whether relevant – If relevant whether excluded as opinion evidence. Evidence Act 1995 (NSW), ss 55, 76.', 'bench': ['GLEESON', 'GAUDRON', 'GUMMOW', 'KIRBY', 'HAYNE'], 'judgmentAuthors': [{'name': 'GLEESON, GAUDRON, GUMMOW, HAYNE', 'type': 'Separate judgment', 'totalCitations': 190, 'paragraphs': '1-17', 'cita

In [14]:
import json
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
from collections import defaultdict

data = backup_cases_all

# Add relativeCitation on the fly if not present
for case in data.values():
    judgments = case['judgmentAuthors']
    sorted_j  = sorted(judgments, key=lambda j: j.get('totalCitations', 0), reverse=True)
    top       = sorted_j[0].get('totalCitations', 0) if sorted_j else 0
    for rank, j in enumerate(sorted_j, start=1):
        j['citationRank']     = rank
        j['relativeCitation'] = round(j.get('totalCitations', 0) / top, 4) if top > 0 else 0
    case['judgmentAuthors'] = sorted_j

gummow_together, gummow_alone = [], []
hayne_together,  hayne_alone  = [], []
both_together,   both_alone   = [], []

for case in data.values():
    for j in case['judgmentAuthors']:
        names = [n.strip() for n in j['name'].split(',')]
        has_g = 'GUMMOW' in names
        has_h = 'HAYNE'  in names
        rc    = j['relativeCitation']

        if has_g and has_h:
            both_together.append(rc)
            gummow_together.append(rc)
            hayne_together.append(rc)
        elif has_g:
            gummow_alone.append(rc)
        elif has_h:
            hayne_alone.append(rc)

DARK   = '#333333'
LIGHT  = '#aaaaaa'
RED    = '#cc3333'

def add_stats(ax, data, pos, color):
    """Annotate median and n beneath each box."""
    ax.text(pos, -0.07, f'n={len(data)}', ha='center', va='top',
            fontsize=8, color='#666', transform=ax.get_xaxis_transform())
    ax.text(pos, -0.12, f'med={np.median(data):.2f}', ha='center', va='top',
            fontsize=8, color=color, fontweight='bold',
            transform=ax.get_xaxis_transform())

fig, axes = plt.subplots(1, 3, figsize=(13, 6), sharey=True)
fig.patch.set_facecolor('white')

panels = [
    ('Gummow', gummow_alone,  None),
    ('Hayne',  hayne_alone,   None),
    ('Gummow & Hayne\ntogether', both_together, None),
]

for ax, (title, tog, alone) in zip(axes, panels):
    ax.set_facecolor('#f8f8f8')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.yaxis.grid(True, linestyle='--', alpha=0.5, zorder=0)
    ax.set_axisbelow(True)

    if alone is not None:
        bp = ax.boxplot(
            [tog, alone],
            positions=[1, 2],
            widths=0.5,
            patch_artist=True,
            medianprops=dict(color=RED, linewidth=2),
            whiskerprops=dict(color='#555'),
            capprops=dict(color='#555'),
            flierprops=dict(marker='o', markersize=3, alpha=0.4,
                            markerfacecolor='#999', markeredgecolor='none'),
        )
        bp['boxes'][0].set_facecolor(DARK)
        bp['boxes'][0].set_alpha(0.75)
        bp['boxes'][1].set_facecolor(LIGHT)
        bp['boxes'][1].set_alpha(0.75)

        ax.set_xticks([1, 2])
        ax.set_xticklabels(['Together', 'Alone'], fontsize=10)
        add_stats(ax, tog,   1, DARK)
        add_stats(ax, alone, 2, '#777')
    else:
        bp = ax.boxplot(
            [tog],
            positions=[1.5],
            widths=0.5,
            patch_artist=True,
            medianprops=dict(color=RED, linewidth=2),
            whiskerprops=dict(color='#555'),
            capprops=dict(color='#555'),
            flierprops=dict(marker='o', markersize=3, alpha=0.4,
                            markerfacecolor='#999', markeredgecolor='none'),
        )
        bp['boxes'][0].set_facecolor(DARK)
        bp['boxes'][0].set_alpha(0.75)
        ax.set_xticks([1.5])
        ax.set_xticklabels(['Together'], fontsize=10)
        add_stats(ax, tog, 1.5, DARK)
        ax.set_xlim(0.8, 2.2)

    ax.set_title(title, fontsize=12, fontweight='bold', pad=10)
    ax.set_ylim(-0.05, 1.1)

axes[0].set_ylabel('Relative citation score', fontsize=11)

fig.suptitle('Relative Citation Score — Writing Together vs. Alone',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('images/citation_boxplots.png', dpi=150,
            bbox_inches='tight', facecolor='white')
print("Saved: citation_boxplots.png")

Saved: citation_boxplots.png


In [16]:
for case in data.values():
    sorted_j = sorted(case['judgmentAuthors'], key=lambda j: j.get('totalCitations', 0), reverse=True)
    top = sorted_j[0].get('totalCitations', 0) if sorted_j else 0
    for rank, j in enumerate(sorted_j, start=1):
        j['citationRank'] = rank
        j['relativeCitation'] = round(j.get('totalCitations', 0) / top, 4) if top > 0 else 0
    case['judgmentAuthors'] = sorted_j
 
judge_rc       = defaultdict(list)
gh_together_rc = []
 
for case in data.values():
    for j in case['judgmentAuthors']:
        names = [n.strip() for n in j['name'].split(',')]
        rc    = j['relativeCitation']
        if 'GUMMOW' in names and 'HAYNE' in names:
            gh_together_rc.append(rc)
        for name in names:
            if name not in ('GUMMOW', 'HAYNE'):
                judge_rc[name].append(rc)
 
JUDGES = sorted(judge_rc.keys())
GH     = '#1a5276'
DARK   = '#333333'
RED    = '#cc3333'
 
all_panels = [('Gummow\n& Hayne', gh_together_rc, GH)] + \
             [(j.capitalize(), judge_rc[j], DARK) for j in JUDGES]
 
n_total = len(all_panels)
n_cols  = (n_total + 1) // 2
fig, axes = plt.subplots(2, n_cols, figsize=(3 * n_cols, 12), sharey=True)
fig.patch.set_facecolor('white')
axes_flat = axes.flatten()
 
def draw_box(ax, vals, color, title):
    ax.set_facecolor('#f8f8f8')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.yaxis.grid(True, linestyle='--', alpha=0.4, zorder=0)
    ax.set_axisbelow(True)
 
    bp = ax.boxplot(
        [vals], positions=[1], widths=0.5, patch_artist=True,
        medianprops=dict(color=RED, linewidth=2),
        whiskerprops=dict(color='#555'),
        capprops=dict(color='#555'),
        flierprops=dict(marker='o', markersize=2.5, alpha=0.35,
                        markerfacecolor='#999', markeredgecolor='none'),
    )
    bp['boxes'][0].set_facecolor(color)
    bp['boxes'][0].set_alpha(0.8)
 
    ax.set_xticks([1])
    ax.set_xticklabels([''], fontsize=8.5)
    ax.set_xlim(0.5, 1.5)
    ax.set_ylim(-0.05, 1.1)
    ax.set_title(title, fontsize=10, fontweight='bold', pad=8)
    ax.text(1, -0.07, f'n={len(vals)}', ha='center', va='top',
            fontsize=8, color='#777', transform=ax.get_xaxis_transform())
    ax.text(1, -0.13, f'med={np.median(vals):.2f}', ha='center', va='top',
            fontsize=8, color='#333', fontweight='bold',
            transform=ax.get_xaxis_transform())
 
for i, (title, vals, color) in enumerate(all_panels):
    draw_box(axes_flat[i], vals, color, title)
 
# Hide any unused axes
for j in range(len(all_panels), len(axes_flat)):
    axes_flat[j].set_visible(False)
 
# y-axis label only on leftmost of each row
for row in range(2):
    axes[row][0].set_ylabel('Relative citation score', fontsize=11)
 
fig.suptitle('Relative Citation Score: G+H together vs. each other judge',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('images/citation_all_judges.png', dpi=150,
            bbox_inches='tight', facecolor='white')
print("Saved.")

Saved.


In [17]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
from collections import defaultdict

data

TOP99 = ['[1998] HCA 28','[2003] HCA 2','[2003] HCA 22','[2001] HCA 30','[2009] HCA 27','[2005] HCA 25','[2010] HCA 16','[1998] HCA 11','[2000] HCA 63','[2006] HCA 63','[1999] HCA 14','[2001] HCA 17','[2000] HCA 54','[1998] HCA 57','[2010] HCA 45','[2004] HCA 52','[2009] HCA 41','[2001] HCA 64','[1999] HCA 21','[2000] HCA 57','[2011] HCA 39','[2000] HCA 47','[2010] HCA 28','[2007] HCA 22','[2010] HCA 1','[2005] HCA 24','[2003] HCA 6','[2000] HCA 48','[2006] HCA 46','[2010] HCA 4','[1998] HCA 67','[1999] HCA 54','[2000] HCA 40','[2008] HCA 31','[1999] HCA 29','[2011] HCA 49','[2011] HCA 4','[2002] HCA 11','[2001] HCA 63','[2004] HCA 35','[2002] HCA 53','[2006] HCA 27','[2000] HCA 41','[2002] HCA 36','[1999] HCA 27','[1998] HCA 27','[1999] HCA 26','[1999] HCA 66','[2001] HCA 22','[1999] HCA 67','[2002] HCA 35','[2011] HCA 48','[2011] HCA 13','[2010] HCA 41','[1999] HCA 9','[1998] HCA 55','[1998] HCA 30','[2011] HCA 21','[1999] HCA 36','[2007] HCA 35','[2005] HCA 12','[2011] HCA 1','[2005] HCA 81','[2008] HCA 36','[1999] HCA 18','[2000] HCA 5','[2010] HCA 23','[2004] HCA 45','[2001] HCA 52','[2000] HCA 12','[2001] HCA 44','[2009] HCA 25','[1999] HCA 10','[2002] HCA 49','[2011] HCA 34','[2001] HCA 59','[2002] HCA 6','[2003] HCA 71','[2011] HCA 11','[1998] HCA 69','[2001] HCA 29','[2007] HCA 30','[2010] HCA 48','[2002] HCA 46','[2002] HCA 54','[2004] HCA 60','[2000] HCA 3','[2005] HCA 10','[2010] HCA 32','[2001] HCA 18','[1999] HCA 37','[2002] HCA 28','[1999] HCA 6','[1998] HCA 1','[2004] HCA 46','[2005] HCA 62','[2006] HCA 55','[2011] HCA 10','[2004] HCA 37','[2003] HCA 1']

judge_rc       = defaultdict(list)
gh_together_rc = []

for citation in TOP99:
    if citation not in data:
        continue
    case = data[citation]
    for j in case['judgmentAuthors']:
        names = [n.strip() for n in j['name'].split(',')]
        rc    = j.get('totalCitations', 0)
        if 'GUMMOW' in names and 'HAYNE' in names:
            gh_together_rc.append(rc)
        for name in names:
            if name not in ('GUMMOW', 'HAYNE'):
                judge_rc[name].append(rc)

JUDGES = sorted(judge_rc.keys())
GH   = '#1a5276'
DARK = '#333333'
RED  = '#cc3333'

all_panels = [('Gummow\n& Hayne', gh_together_rc, GH)] + \
             [(j.capitalize(), judge_rc[j], DARK) for j in JUDGES]

n_total = len(all_panels)
n_cols  = (n_total + 1) // 2
fig, axes = plt.subplots(2, n_cols, figsize=(3 * n_cols, 12), sharey=True)
fig.patch.set_facecolor('white')
axes_flat = axes.flatten()

def draw_box(ax, vals, color, title):
    ax.set_facecolor('#f8f8f8')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.yaxis.grid(True, linestyle='--', alpha=0.4, zorder=0)
    ax.set_axisbelow(True)

    bp = ax.boxplot(
        [vals], positions=[1], widths=0.5, patch_artist=True,
        medianprops=dict(color=RED, linewidth=2),
        whiskerprops=dict(color='#555'),
        capprops=dict(color='#555'),
        flierprops=dict(marker='o', markersize=2.5, alpha=0.35,
                        markerfacecolor='#999', markeredgecolor='none'),
    )
    bp['boxes'][0].set_facecolor(color)
    bp['boxes'][0].set_alpha(0.8)

    ax.set_xticks([1])
    ax.set_xticklabels([''], fontsize=8.5)
    ax.set_xlim(0.5, 1.5)
    ax.set_title(title, fontsize=10, fontweight='bold', pad=8)
    ax.text(1, -0.04, f'n={len(vals)}', ha='center', va='top',
            fontsize=8, color='#777', transform=ax.get_xaxis_transform())
    ax.text(1, -0.10, f'med={np.median(vals):.0f}', ha='center', va='top',
            fontsize=8, color='#333', fontweight='bold',
            transform=ax.get_xaxis_transform())

for i, (title, vals, color) in enumerate(all_panels):
    draw_box(axes_flat[i], vals, color, title)

for j in range(len(all_panels), len(axes_flat)):
    axes_flat[j].set_visible(False)

for row in range(2):
    axes[row][0].set_ylabel('Raw citations', fontsize=11)

fig.suptitle('Raw Citation Count: G+H together vs. each other judge\n(top 99 most cited cases)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('images/citation_raw_top99.png', dpi=150,
            bbox_inches='tight', facecolor='white')
print("Saved.")

Saved.


In [18]:
print(len(backup_cases_all))

781


In [23]:
import json
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
from collections import defaultdict

data

TOP99 = ['[1998] HCA 28','[2003] HCA 2','[2003] HCA 22','[2001] HCA 30','[2009] HCA 27','[2005] HCA 25','[2010] HCA 16','[1998] HCA 11','[2000] HCA 63','[2006] HCA 63','[1999] HCA 14','[2001] HCA 17','[2000] HCA 54','[1998] HCA 57','[2010] HCA 45','[2004] HCA 52','[2009] HCA 41','[2001] HCA 64','[1999] HCA 21','[2000] HCA 57','[2011] HCA 39','[2000] HCA 47','[2010] HCA 28','[2007] HCA 22','[2010] HCA 1','[2005] HCA 24','[2003] HCA 6','[2000] HCA 48','[2006] HCA 46','[2010] HCA 4','[1998] HCA 67','[1999] HCA 54','[2000] HCA 40','[2008] HCA 31','[1999] HCA 29','[2011] HCA 49','[2011] HCA 4','[2002] HCA 11','[2001] HCA 63','[2004] HCA 35','[2002] HCA 53','[2006] HCA 27','[2000] HCA 41','[2002] HCA 36','[1999] HCA 27','[1998] HCA 27','[1999] HCA 26','[1999] HCA 66','[2001] HCA 22','[1999] HCA 67','[2002] HCA 35','[2011] HCA 48','[2011] HCA 13','[2010] HCA 41','[1999] HCA 9','[1998] HCA 55','[1998] HCA 30','[2011] HCA 21','[1999] HCA 36','[2007] HCA 35','[2005] HCA 12','[2011] HCA 1','[2005] HCA 81','[2008] HCA 36','[1999] HCA 18','[2000] HCA 5','[2010] HCA 23','[2004] HCA 45','[2001] HCA 52','[2000] HCA 12','[2001] HCA 44','[2009] HCA 25','[1999] HCA 10','[2002] HCA 49','[2011] HCA 34','[2001] HCA 59','[2002] HCA 6','[2003] HCA 71','[2011] HCA 11','[1998] HCA 69','[2001] HCA 29','[2007] HCA 30','[2010] HCA 48','[2002] HCA 46','[1997] HCA 56','[2002] HCA 54','[2004] HCA 60','[2000] HCA 3','[2005] HCA 10','[2010] HCA 32','[2001] HCA 18','[1999] HCA 37','[2002] HCA 28','[1999] HCA 6','[1998] HCA 1','[2004] HCA 46','[2005] HCA 62','[2006] HCA 55','[2011] HCA 10','[2004] HCA 37','[2003] HCA 1']

def gh_wrote_together(case):
    return any(
        'GUMMOW' in [n.strip() for n in j['name'].split(',')]
        and 'HAYNE' in [n.strip() for n in j['name'].split(',')]
        for j in case['judgmentAuthors']
    )

BAR_COLOR = '#333333'
LIGHT     = '#aaaaaa'
RED       = '#cc3333'

# ══════════════════════════════════════════════════════════
# GRAPH 1 — G+H together per decile
# ══════════════════════════════════════════════════════════
decile_together = []
decile_not      = []
decile_labels   = []

for d in range(10):
    decile = TOP99[d*10:(d+1)*10]
    tog = sum(1 for c in decile if c in data and gh_wrote_together(data[c]))
    decile_together.append(tog)
    decile_not.append(10 - tog)
    decile_labels.append(f'{d*10+1}–{(d+1)*10}')

x = np.arange(len(decile_labels))

fig, ax = plt.subplots(figsize=(11, 5))
fig.patch.set_facecolor('white')
ax.set_facecolor('#f8f8f8')

b1 = ax.bar(x, decile_together, color=BAR_COLOR, edgecolor='white', linewidth=0.6,
            zorder=3, label='Wrote together')
b2 = ax.bar(x, decile_not, bottom=decile_together, color=LIGHT, edgecolor='white',
            linewidth=0.6, zorder=3, label='Did not write together')

for i, (tog, notg) in enumerate(zip(decile_together, decile_not)):
    ax.text(i, tog / 2, str(tog), ha='center', va='center',
            fontsize=10, color='white', fontweight='bold', zorder=4)
    ax.text(i, tog + notg / 2, str(notg), ha='center', va='center',
            fontsize=10, color='#555', fontweight='bold', zorder=4)

ax.set_xticks(x)
ax.set_xticklabels([f'#{l}' for l in decile_labels], fontsize=10)
ax.set_xlabel('Citation rank (top 100 cases)', fontsize=11)
ax.set_ylabel('Number of cases', fontsize=11)
ax.set_title('Gummow & Hayne — Co-authorship across citation deciles (top 100 cases)',
             fontsize=13, fontweight='bold', pad=12)
ax.set_ylim(0, 12)
ax.yaxis.grid(True, linestyle='--', alpha=0.5, zorder=0)
ax.set_axisbelow(True)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.legend(fontsize=10, frameon=True, facecolor='white', edgecolor='#aaa')

plt.tight_layout()
plt.savefig('images/top99_decile.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.close()
print("Saved: top99_decile.png")

# ══════════════════════════════════════════════════════════
# GRAPH 2 — together vs not by tag (top 99 only)
# ════════════════════════════════════════════════════════
together     = defaultdict(int)
not_together = defaultdict(int)
tag_totals   = defaultdict(int)

for citation in TOP99:
    if citation not in data:
        continue
    case = data[citation]
    tags = case['law_areas']
    gh   = gh_wrote_together(case)
    for tag in tags:
        tag_totals[tag] += 1
        if gh:
            together[tag] += 1
        else:
            not_together[tag] += 1

# Sort by together % descending
tags_sorted = sorted(tag_totals.keys(),
                     key=lambda t: together[t] / tag_totals[t], reverse=True)
tog  = np.array([together[t]     for t in tags_sorted])
notg = np.array([not_together[t] for t in tags_sorted])
tots = tog + notg
labels = [t.replace(' & ', '\n& ').replace(', Planning', ',\nPlanning') for t in tags_sorted]
x = np.arange(len(tags_sorted))

fig, ax = plt.subplots(figsize=(14, 6))
fig.patch.set_facecolor('white')
ax.set_facecolor('#f8f8f8')

ax.bar(x, tog,  color=BAR_COLOR, label='Wrote together',          edgecolor='white', linewidth=0.6, zorder=3)
ax.bar(x, notg, color=LIGHT,     label='Did not write together',   edgecolor='white', linewidth=0.6,
       bottom=tog, zorder=3)

for i, (t, total) in enumerate(zip(tog, tots)):
    if t > 0:
        pct = t / total
        ax.text(i, t / 2, f'{pct:.0%}', ha='center', va='center',
                fontsize=8, color='white', fontweight='bold', zorder=4)
for i, total in enumerate(tots):
    ax.text(i, total + 0.1, f'n={total}', ha='center', va='bottom',
            fontsize=7, color='#555', zorder=4)

ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=40, ha='right', fontsize=8.5)
ax.set_ylabel('Number of cases', fontsize=11)
ax.set_title('Gummow & Hayne — Wrote together vs. not, by tag (top 99 cases)\n(sorted by co-authorship rate)',
             fontsize=13, fontweight='bold', pad=12)
ax.yaxis.grid(True, linestyle='--', alpha=0.5, zorder=0)
ax.set_axisbelow(True)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.legend(fontsize=10, frameon=True, facecolor='white', edgecolor='#aaa')

plt.tight_layout()
plt.savefig('images/top99_tags.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.close()
print("Saved: top99_tags.png")

Saved: top99_decile.png
Saved: top99_tags.png


In [24]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
from collections import defaultdict

data = backup_cases_all

TOP99 = ['[1998] HCA 28','[2003] HCA 2','[2003] HCA 22','[2001] HCA 30','[2009] HCA 27','[2005] HCA 25','[2010] HCA 16','[1998] HCA 11','[2000] HCA 63','[2006] HCA 63','[1999] HCA 14','[2001] HCA 17','[2000] HCA 54','[1998] HCA 57','[2010] HCA 45','[2004] HCA 52','[2009] HCA 41','[2001] HCA 64','[1999] HCA 21','[2000] HCA 57','[2011] HCA 39','[2000] HCA 47','[2010] HCA 28','[2007] HCA 22','[2010] HCA 1','[2005] HCA 24','[2003] HCA 6','[2000] HCA 48','[2006] HCA 46','[2010] HCA 4','[1998] HCA 67','[1999] HCA 54','[2000] HCA 40','[2008] HCA 31','[1999] HCA 29','[2011] HCA 49','[2011] HCA 4','[2002] HCA 11','[2001] HCA 63','[2004] HCA 35','[2002] HCA 53','[2006] HCA 27','[2000] HCA 41','[2002] HCA 36','[1999] HCA 27','[1998] HCA 27','[1999] HCA 26','[1999] HCA 66','[2001] HCA 22','[1999] HCA 67','[2002] HCA 35','[2011] HCA 48','[2011] HCA 13','[2010] HCA 41','[1999] HCA 9','[1998] HCA 55','[1998] HCA 30','[2011] HCA 21','[1999] HCA 36','[2007] HCA 35','[2005] HCA 12','[2011] HCA 1','[2005] HCA 81','[2008] HCA 36','[1999] HCA 18','[2000] HCA 5','[2010] HCA 23','[2004] HCA 45','[2001] HCA 52','[2000] HCA 12','[2001] HCA 44','[2009] HCA 25','[1999] HCA 10','[2002] HCA 49','[2011] HCA 34','[2001] HCA 59','[2002] HCA 6','[2003] HCA 71','[2011] HCA 11','[1998] HCA 69','[2001] HCA 29','[2007] HCA 30','[2010] HCA 48','[2002] HCA 46','[1997] HCA 56','[2002] HCA 54','[2004] HCA 60','[2000] HCA 3','[2005] HCA 10','[2010] HCA 32','[2001] HCA 18','[1999] HCA 37','[2002] HCA 28','[1999] HCA 6','[1998] HCA 1','[2004] HCA 46','[2005] HCA 62','[2006] HCA 55','[2011] HCA 10','[2004] HCA 37','[2003] HCA 1']

def gh_wrote_together(case):
    return any(
        'GUMMOW' in [n.strip() for n in j['name'].split(',')]
        and 'HAYNE' in [n.strip() for n in j['name'].split(',')]
        for j in case['judgmentAuthors']
    )

together     = defaultdict(int)
not_together = defaultdict(int)
tag_totals   = defaultdict(int)

for citation in TOP99:
    if citation not in data:
        continue
    case = data[citation]
    tag  = case['law_areas'][0]   # first tag only
    if tag in ('Immigration & Refugees', 'Administrative Law'):
        tag = 'Admin Law & Immigration'
    gh   = gh_wrote_together(case)
    tag_totals[tag] += 1
    if gh:
        together[tag] += 1
    else:
        not_together[tag] += 1

# Sort by together % descending
tags_sorted = sorted(tag_totals.keys(),
                     key=lambda t: together[t] / tag_totals[t], reverse=True)
tog    = np.array([together[t]     for t in tags_sorted])
notg   = np.array([not_together[t] for t in tags_sorted])
tots   = tog + notg
labels = [t.replace(' & ', '\n& ').replace(', Planning', ',\nPlanning') for t in tags_sorted]
x      = np.arange(len(tags_sorted))

fig, ax = plt.subplots(figsize=(14, 6))
fig.patch.set_facecolor('white')
ax.set_facecolor('#f8f8f8')

ax.bar(x, tog,  color='#333333', label='Wrote together',        edgecolor='white', linewidth=0.6, zorder=3)
ax.bar(x, notg, color='#aaaaaa', label='Did not write together', edgecolor='white', linewidth=0.6,
       bottom=tog, zorder=3)

for i, (t, total) in enumerate(zip(tog, tots)):
    if t > 0:
        ax.text(i, t / 2, f'{t/total:.0%}', ha='center', va='center',
                fontsize=8, color='white', fontweight='bold', zorder=4)
for i, total in enumerate(tots):
    ax.text(i, total + 0.1, f'n={total}', ha='center', va='bottom',
            fontsize=7, color='#555', zorder=4)

ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=40, ha='right', fontsize=8.5)
ax.set_ylabel('Number of cases', fontsize=11)
ax.set_title('Gummow & Hayne — Wrote together vs. not, by primary tag (top 100 cases)\n'
             '(sorted by co-authorship rate; one tag per case)',
             fontsize=13, fontweight='bold', pad=12)
ax.yaxis.grid(True, linestyle='--', alpha=0.5, zorder=0)
ax.set_axisbelow(True)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.legend(fontsize=10, frameon=True, facecolor='white', edgecolor='#aaa')

plt.tight_layout()
plt.savefig('images/top99_tags_one_tag.png', dpi=150, bbox_inches='tight', facecolor='white')
print("Saved: top99_tags_one_tag.png")

Saved: top99_tags_one_tag.png


In [25]:
import json
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
from collections import defaultdict

data = backup_cases_all

TOP99 = ['[1998] HCA 28','[2003] HCA 2','[2003] HCA 22','[2001] HCA 30','[2009] HCA 27','[2005] HCA 25','[2010] HCA 16','[1998] HCA 11','[2000] HCA 63','[2006] HCA 63','[1999] HCA 14','[2001] HCA 17','[2000] HCA 54','[1998] HCA 57','[2010] HCA 45','[2004] HCA 52','[2009] HCA 41','[2001] HCA 64','[1999] HCA 21','[2000] HCA 57','[2011] HCA 39','[2000] HCA 47','[2010] HCA 28','[2007] HCA 22','[2010] HCA 1','[2005] HCA 24','[2003] HCA 6','[2000] HCA 48','[2006] HCA 46','[2010] HCA 4','[1998] HCA 67','[1999] HCA 54','[2000] HCA 40','[2008] HCA 31','[1999] HCA 29','[2011] HCA 49','[2011] HCA 4','[2002] HCA 11','[2001] HCA 63','[2004] HCA 35','[2002] HCA 53','[2006] HCA 27','[2000] HCA 41','[2002] HCA 36','[1999] HCA 27','[1998] HCA 27','[1999] HCA 26','[1999] HCA 66','[2001] HCA 22','[1999] HCA 67','[2002] HCA 35','[2011] HCA 48','[2011] HCA 13','[2010] HCA 41','[1999] HCA 9','[1998] HCA 55','[1998] HCA 30','[2011] HCA 21','[1999] HCA 36','[2007] HCA 35','[2005] HCA 12','[2011] HCA 1','[2005] HCA 81','[2008] HCA 36','[1999] HCA 18','[2000] HCA 5','[2010] HCA 23','[2004] HCA 45','[2001] HCA 52','[2000] HCA 12','[2001] HCA 44','[2009] HCA 25','[1999] HCA 10','[2002] HCA 49','[2011] HCA 34','[2001] HCA 59','[2002] HCA 6','[2003] HCA 71','[2011] HCA 11','[1998] HCA 69','[2001] HCA 29','[2007] HCA 30','[2010] HCA 48','[2002] HCA 46','[1997] HCA 56','[2002] HCA 54','[2004] HCA 60','[2000] HCA 3','[2005] HCA 10','[2010] HCA 32','[2001] HCA 18','[1999] HCA 37','[2002] HCA 28','[1999] HCA 6','[1998] HCA 1','[2004] HCA 46','[2005] HCA 62','[2006] HCA 55','[2011] HCA 10','[2004] HCA 37','[2003] HCA 1']

# Add relativeCitation on the fly
for case in data.values():
    sorted_j = sorted(case['judgmentAuthors'], key=lambda j: j.get('totalCitations', 0), reverse=True)
    top = sorted_j[0].get('totalCitations', 0) if sorted_j else 0
    for rank, j in enumerate(sorted_j, start=1):
        j['citationRank'] = rank
        j['relativeCitation'] = round(j.get('totalCitations', 0) / top, 4) if top > 0 else 0
    case['judgmentAuthors'] = sorted_j

judge_rc       = defaultdict(list)
gh_together_rc = []

for citation in TOP99:
    if citation not in data:
        continue
    case = data[citation]
    for j in case['judgmentAuthors']:
        names = [n.strip() for n in j['name'].split(',')]
        rc    = j['relativeCitation']
        if 'GUMMOW' in names and 'HAYNE' in names:
            gh_together_rc.append(rc)
        for name in names:
            if name not in ('GUMMOW', 'HAYNE'):
                judge_rc[name].append(rc)

JUDGES = sorted(judge_rc.keys())
GH   = '#1a5276'
DARK = '#333333'
RED  = '#cc3333'

all_panels = [('Gummow\n& Hayne', gh_together_rc, GH)] + \
             [(j.capitalize(), judge_rc[j], DARK) for j in JUDGES]

n_total = len(all_panels)
n_cols  = (n_total + 1) // 2
fig, axes = plt.subplots(2, n_cols, figsize=(3 * n_cols, 12), sharey=True)
fig.patch.set_facecolor('white')
axes_flat = axes.flatten()

def draw_box(ax, vals, color, title):
    ax.set_facecolor('#f8f8f8')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.yaxis.grid(True, linestyle='--', alpha=0.4, zorder=0)
    ax.set_axisbelow(True)

    bp = ax.boxplot(
        [vals], positions=[1], widths=0.5, patch_artist=True,
        medianprops=dict(color=RED, linewidth=2),
        whiskerprops=dict(color='#555'),
        capprops=dict(color='#555'),
        flierprops=dict(marker='o', markersize=4, alpha=0.9,
                        markerfacecolor='#222', markeredgecolor='#222'),
    )
    bp['boxes'][0].set_facecolor(color)
    bp['boxes'][0].set_alpha(0.8)

    ax.set_xticks([1])
    ax.set_xticklabels([''], fontsize=8.5)
    ax.set_xlim(0.5, 1.5)
    ax.set_ylim(-0.05, 1.1)
    ax.set_title(title, fontsize=10, fontweight='bold', pad=8)
    ax.text(1, -0.07, f'n={len(vals)}', ha='center', va='top',
            fontsize=8, color='#777', transform=ax.get_xaxis_transform())
    ax.text(1, -0.13, f'med={np.median(vals):.2f}', ha='center', va='top',
            fontsize=8, color='#333', fontweight='bold',
            transform=ax.get_xaxis_transform())

for i, (title, vals, color) in enumerate(all_panels):
    draw_box(axes_flat[i], vals, color, title)

for j in range(len(all_panels), len(axes_flat)):
    axes_flat[j].set_visible(False)

for row in range(2):
    axes[row][0].set_ylabel('Relative citation score', fontsize=11)

fig.suptitle('Relative Citation Score: G+H together vs. each other judge\n(top 99 most cited cases)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('images/citation_relative_top99.png', dpi=150,
            bbox_inches='tight', facecolor='white')
print("Saved.")

Saved.


In [26]:
import json
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
from collections import defaultdict

data = backup_cases_all

yearly = defaultdict(lambda: {'together': 0, 'total': 0})

for citation, case in data.items():
    year = int(citation.split(']')[0].replace('[', '').strip())
    yearly[year]['total'] += 1
    if any('GUMMOW' in [n.strip() for n in j['name'].split(',')]
           and 'HAYNE' in [n.strip() for n in j['name'].split(',')]
           for j in case['judgmentAuthors']):
        yearly[year]['together'] += 1

years   = sorted(yearly)
tog     = [yearly[y]['together'] for y in years]
totals  = [yearly[y]['total']    for y in years]
rate    = [yearly[y]['together'] / yearly[y]['total'] for y in years]

BAR_COLOR  = '#333333'
LIGHT      = '#aaaaaa'
LINE_COLOR = '#cc3333'

fig, ax1 = plt.subplots(figsize=(13, 6))
fig.patch.set_facecolor('white')
ax1.set_facecolor('#f8f8f8')

x = np.arange(len(years))
ax1.bar(x, totals, color=LIGHT,     edgecolor='white', linewidth=0.6, zorder=3, label='Total cases')
ax1.bar(x, tog,    color=BAR_COLOR, edgecolor='white', linewidth=0.6, zorder=4, label='G+H wrote together')

ax1.set_xticks(x)
ax1.set_xticklabels(years, fontsize=10)
ax1.set_ylabel('Number of cases', fontsize=11)
ax1.set_xlabel('Year', fontsize=11)
ax1.yaxis.grid(True, linestyle='--', alpha=0.5, zorder=0)
ax1.set_axisbelow(True)
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)

# Rate line on secondary axis
ax2 = ax1.twinx()
ax2.plot(x, rate, color=LINE_COLOR, linewidth=2, marker='o',
         markersize=5, zorder=5, label='Co-authorship rate')
ax2.set_ylabel('Co-authorship rate', fontsize=11, color=LINE_COLOR)
ax2.tick_params(axis='y', labelcolor=LINE_COLOR)
ax2.set_ylim(0, 1)
ax2.yaxis.set_major_formatter(matplotlib.ticker.PercentFormatter(xmax=1))
ax2.spines['top'].set_visible(False)

# Combined legend
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2,
           fontsize=10, frameon=True, facecolor='white', edgecolor='#aaa', loc='upper left')

ax1.set_title('Gummow & Hayne — Co-authorship over time', fontsize=13, fontweight='bold', pad=12)

plt.tight_layout()
plt.savefig('images/gh_over_time.png', dpi=150, bbox_inches='tight', facecolor='white')
print("Saved.")

Saved.


In [27]:
import json
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
from collections import defaultdict

data= backup_cases_all

yearly_tags = defaultdict(lambda: defaultdict(int))
for citation, case in data.items():
    year = int(citation.split(']')[0].replace('[', '').strip())
    if any('GUMMOW' in [n.strip() for n in j['name'].split(',')]
           and 'HAYNE' in [n.strip() for n in j['name'].split(',')]
           for j in case['judgmentAuthors']):
        tag = case['law_areas'][0]
        if tag in ('Administrative Law', 'Immigration & Refugees'):
            tag = 'Admin Law & Immigration'
        yearly_tags[year][tag] += 1

years = sorted(yearly_tags)

# Order tags by total frequency descending (for legend clarity)
tag_totals = defaultdict(int)
for y in yearly_tags.values():
    for tag, count in y.items():
        tag_totals[tag] += count
all_tags = sorted(tag_totals, key=lambda t: -tag_totals[t])

# Assign colors — use tab20 for enough distinct colors
cmap   = plt.cm.tab20
colors = {tag: cmap(i / len(all_tags)) for i, tag in enumerate(all_tags)}

x      = np.arange(len(years))
fig, ax = plt.subplots(figsize=(14, 7))
fig.patch.set_facecolor('white')
ax.set_facecolor('#f8f8f8')

bottoms = np.zeros(len(years))
for tag in all_tags:
    vals = np.array([yearly_tags[y].get(tag, 0) for y in years])
    ax.bar(x, vals, bottom=bottoms, color=colors[tag],
           edgecolor='white', linewidth=0.4, zorder=3, label=tag)
    bottoms += vals

ax.set_xticks(x)
ax.set_xticklabels(years, fontsize=10)
ax.set_ylabel('Number of cases', fontsize=11)
ax.set_xlabel('Year', fontsize=11)
ax.set_title('Gummow & Hayne — Cases written together by tag over time',
             fontsize=13, fontweight='bold', pad=12)
ax.yaxis.grid(True, linestyle='--', alpha=0.4, zorder=0)
ax.set_axisbelow(True)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.legend(fontsize=8.5, frameon=True, facecolor='white', edgecolor='#aaa',
          bbox_to_anchor=(1.01, 1), loc='upper left', title='Tag', title_fontsize=9)

plt.tight_layout()
plt.savefig('images/gh_over_time_tags.png', dpi=150,
            bbox_inches='tight', facecolor='white')
print("Saved.")

Saved.


In [28]:
import json
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker
import numpy as np
from collections import defaultdict

data = backup_cases_all

def merge_tag(tag):
    if tag in ('Administrative Law', 'Immigration & Refugees'):
        return 'Admin Law & Immigration'
    return tag

yearly_together = defaultdict(lambda: defaultdict(int))
yearly_total    = defaultdict(lambda: defaultdict(int))

for citation, case in data.items():
    year = int(citation.split(']')[0].replace('[', '').strip())
    tag  = merge_tag(case['law_areas'][0])
    yearly_total[year][tag] += 1
    if any('GUMMOW' in [n.strip() for n in j['name'].split(',')]
           and 'HAYNE' in [n.strip() for n in j['name'].split(',')]
           for j in case['judgmentAuthors']):
        yearly_together[year][tag] += 1

years    = sorted(yearly_total.keys())
all_tags = sorted({t for y in yearly_total.values() for t in y})
all_tags = [t for t in all_tags
            if sum(yearly_total[y].get(t, 0) for y in years) >= 5]
all_tags = sorted(all_tags)

n_tags = len(all_tags)
n_cols = 4
n_rows = (n_tags + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, n_rows * 2.2),
                         sharey=True, sharex=False)
fig.patch.set_facecolor('white')
axes_flat = axes.flatten()

BAR_COLOR = '#333333'
LIGHT     = '#dddddd'

for i, tag in enumerate(all_tags):
    ax = axes_flat[i]
    ax.set_facecolor('#f8f8f8')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_visible(False)

    tag_years = [y for y in years if yearly_total[y].get(tag, 0) > 0]
    tog   = np.array([yearly_together[y].get(tag, 0) for y in tag_years], dtype=float)
    total = np.array([yearly_total[y].get(tag, 0)    for y in tag_years], dtype=float)
    rate  = tog / total  # proportion

    x = np.arange(len(tag_years))
    ax.bar(x, np.ones(len(tag_years)), color=LIGHT,     edgecolor='white', linewidth=0.3, zorder=2)
    ax.bar(x, rate,                    color=BAR_COLOR,  edgecolor='white', linewidth=0.3, zorder=3)

    # n= total cases below each bar
    for xi, (y, n) in enumerate(zip(tag_years, total.astype(int))):
        ax.text(xi, -0.08, str(n), ha='center', va='top', fontsize=5.5,
                color='#888', transform=ax.get_xaxis_transform())

    ax.set_xticks(x)
    ax.set_xticklabels([str(y)[2:] for y in tag_years], fontsize=6.5, rotation=45, ha='right')
    ax.set_title(tag, fontsize=8.5, fontweight='bold', pad=5)
    ax.set_ylim(0, 1.05)
    ax.yaxis.set_major_formatter(matplotlib.ticker.PercentFormatter(xmax=1))
    ax.tick_params(axis='y', labelsize=6.5)
    ax.yaxis.grid(True, linestyle='--', alpha=0.4, zorder=0)
    ax.set_axisbelow(True)

for j in range(len(all_tags), len(axes_flat)):
    axes_flat[j].set_visible(False)

fig.suptitle('Gummow & Hayne — Co-authorship rate by tag over time\n'
             '(dark = G+H together as % of total cases in that tag/year; n = total cases)',
             fontsize=12, fontweight='bold', y=1.01)
plt.tight_layout(h_pad=1.8, w_pad=1.0)
plt.savefig('images/gh_tag_over_time.png', dpi=150,
            bbox_inches='tight', facecolor='white')
print("Saved.")

Saved.


In [29]:
import json
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
from collections import defaultdict

data = backup_cases_all

# Add relativeCitation on the fly
for case in data.values():
    sorted_j = sorted(case['judgmentAuthors'], key=lambda j: j.get('totalCitations', 0), reverse=True)
    top = sorted_j[0].get('totalCitations', 0) if sorted_j else 0
    for rank, j in enumerate(sorted_j, start=1):
        j['citationRank'] = rank
        j['relativeCitation'] = round(j.get('totalCitations', 0) / top, 4) if top > 0 else 0
    case['judgmentAuthors'] = sorted_j

gummow_all,  hayne_all,  gh_together = [], [], []
gummow_solo, hayne_solo              = [], []

for case in data.values():
    for j in case['judgmentAuthors']:
        names = [n.strip() for n in j['name'].split(',')]
        rc    = j['relativeCitation']
        has_g = 'GUMMOW' in names
        has_h = 'HAYNE'  in names

        if has_g and has_h:
            gh_together.append(rc)
            gummow_all.append(rc)
            hayne_all.append(rc)
        elif has_g:
            gummow_all.append(rc)
            gummow_solo.append(rc)
        elif has_h:
            hayne_all.append(rc)
            hayne_solo.append(rc)

GH   = '#1a5276'
DARK = '#444444'
RED  = '#cc3333'

def make_fig(panels, title, filename):
    fig, axes = plt.subplots(1, 3, figsize=(13, 6), sharey=True)
    fig.patch.set_facecolor('white')

    for ax, (label, vals, color, xlabel) in zip(axes, panels):
        ax.set_facecolor('#f8f8f8')
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.yaxis.grid(True, linestyle='--', alpha=0.5, zorder=0)
        ax.set_axisbelow(True)

        bp = ax.boxplot(
            [vals], positions=[1], widths=0.5, patch_artist=True,
            medianprops=dict(color=RED, linewidth=2),
            whiskerprops=dict(color='#555'),
            capprops=dict(color='#555'),
            flierprops=dict(marker='o', markersize=4, alpha=0.9,
                            markerfacecolor='#222', markeredgecolor='#222'),
        )
        bp['boxes'][0].set_facecolor(color)
        bp['boxes'][0].set_alpha(0.8)

        ax.set_xticks([1])
        ax.set_xticklabels([xlabel], fontsize=10)
        ax.set_xlim(0.5, 1.5)
        ax.set_ylim(-0.05, 1.1)
        ax.set_title(label, fontsize=12, fontweight='bold', pad=10)
        ax.text(1, -0.07, f'n={len(vals)}', ha='center', va='top',
                fontsize=9, color='#777', transform=ax.get_xaxis_transform())
        ax.text(1, -0.13, f'med={np.median(vals):.2f}', ha='center', va='top',
                fontsize=9, color='#333', fontweight='bold',
                transform=ax.get_xaxis_transform())

    axes[0].set_ylabel('Relative citation score', fontsize=11)
    fig.suptitle(title, fontsize=14, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.savefig(filename, dpi=150, bbox_inches='tight', facecolor='white')
    plt.close()
    print(f"Saved: {filename}")

# Graph 1 — all judgments (including joint)
make_fig(
    panels=[
        ('Gummow',          gummow_all,  DARK, 'All judgments'),
        ('Hayne',           hayne_all,   DARK, 'All judgments'),
        ('Gummow & Hayne\ntogether', gh_together, GH, 'Together'),
    ],
    title='Relative Citation Score — All judgments (incl. joint)',
    filename='images/citation_all_judgments.png'
)

# Graph 2 — solo judgments only
make_fig(
    panels=[
        ('Gummow',          gummow_solo, DARK, 'Solo only'),
        ('Hayne',           hayne_solo,  DARK, 'Solo only'),
        ('Gummow & Hayne\ntogether', gh_together, GH, 'Together'),
    ],
    title='Relative Citation Score — Solo judgments vs. G+H together',
    filename='images/citation_solo_judgments.png'
)

Saved: images/citation_all_judgments.png
Saved: images/citation_solo_judgments.png


In [ ]:
# FIX
data = backup_cases_all

rows = []
for citation, case in data.items():
    bench = ', '.join(case.get('bench', []))
    for p in case['paragraphs']:
        rows.append({
            'rank':          None,
            'para_number':   p['para'],
            'case':          citation,
            'judgment_authors': p['judge'],
            'bench':         bench,
            'citations':     p.get('citations', 0),
            'snippet':       p.get('snippet', ''),
        })

# Sort by citations descending, take top 1000
rows.sort(key=lambda x: -x['citations'])
rows = rows[:1000]
for i, r in enumerate(rows, start=1):
    r['rank'] = i

# Print to console
print(f"{'Rank':<5} {'Para':>5} {'Citations':>10}  {'Case':<20} {'Judgment authors':<50} {'Bench'}")
print("-" * 160)
for r in rows:
    print(f"{r['rank']:<5} {r['para_number']:>5} {r['citations']:>10,}  {r['case']:<20} {r['judgment_authors']:<50} {r['bench']}")

"""# Also save as CSV
with open('top1000_paragraphs.csv', 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=['rank','para_number','case','judgment_authors','bench','citations','snippet'])
    writer.writeheader()
    writer.writerows(rows)

print(f"\nSaved: top1000_paragraphs.csv ({len(rows)} rows)")"""